<a href="https://colab.research.google.com/github/poonam-021/explainable-ai-stress-burnout/blob/main/Bio_Physiological_Analysis_of_stress.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

============================================================
WESAD Dataset
============================================================

What this script does:
  1. Loads each subject's .pkl file
  2. Extracts EDA, TEMP (chest) and BVP -> HR (wrist)
  3. Synchronizes all signals to 1-second intervals
  4. Filters labels: Baseline (1) vs Stress (2) only
  5. Saves a clean CSV -> wesad_features.csv (ready for Week 2 TabNet)


WESAD Label Reference:
  
  0 = Not defined / transient
  
  1 = Baseline (Relaxed)
  
  2 = Stress  <-- We want this
  
  3 = Amusement
  
  4 = Meditation


Chest RespiBAN sampling rates:
  
  EDA, TEMP, EMG, Resp -> 700 Hz
  
  ECG                  -> 700 Hz
  
  ACC                  -> 700 Hz


Wrist Empatica E4 sampling rates:
  
  BVP  -> 64 Hz
  
  EDA  -> 4 Hz
  
  TEMP -> 4 Hz
  
  ACC  -> 32 Hz
  
  IBI  -> 1 Hz (already events, not fixed rate)

Label sampling rate: 700 Hz (chest device)
============================================================

In [2]:
import os
import pickle
import numpy as np
import pandas as pd
from scipy.signal import find_peaks, butter, filtfilt
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{\r\n  "username": "divanshityagi",\r\n  "key": "KGAT_2df1af97b16cc05facf506146cbfb0ef"\r\n}'}

In [3]:
import os

os.makedirs('/root/.kaggle', exist_ok=True)
!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [4]:
!kaggle datasets download -d mohamedasem318/wesad-full-dataset
!unzip -qo wesad-full-dataset.zip -d /content/WESAD

Dataset URL: https://www.kaggle.com/datasets/mohamedasem318/wesad-full-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
wesad-full-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [5]:
import os
print(os.listdir("/content/WESAD/WESAD"))

['S10', 'S5', 'S9', 'S16', 'S3', 'S17', 'S8', 'wesad_readme.pdf', 'S6', 'S2', 'S15', 'S7', 'S14', 'S11', 'S4', 'S13']


In [6]:
print(os.listdir("/content/WESAD/WESAD"))

['S10', 'S5', 'S9', 'S16', 'S3', 'S17', 'S8', 'wesad_readme.pdf', 'S6', 'S2', 'S15', 'S7', 'S14', 'S11', 'S4', 'S13']


In [7]:
# ─────────────────────────────────────────────
# CONFIGURATION — Edit this path to your data
# ─────────────────────────────────────────────
WESAD_PATH = "/content/WESAD/WESAD"         # Folder containing S2/, S3/, ... subfolders
OUTPUT_CSV = "wesad_features.csv"

# Subjects in the WESAD dataset (S1 was excluded by researchers)
SUBJECT_IDS = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]

# Sampling rates (Hz)
FS_CHEST  = 700   # Chest device (EDA, TEMP, label)
FS_BVP    = 64    # Wrist BVP (used to compute HR)

# Labels we care about
LABEL_BASELINE = 1
LABEL_STRESS   = 2

In [8]:
# ─────────────────────────────────────────────
# HELPER: Load one subject's .pkl file
# ─────────────────────────────────────────────
def load_subject(subject_id: int) -> dict:
    """Load and return raw data dict for a single subject."""
    pkl_path = os.path.join(WESAD_PATH, f"S{subject_id}", f"S{subject_id}.pkl")
    if not os.path.exists(pkl_path):
        print(f"  [WARNING] File not found: {pkl_path} — skipping.")
        return None
    with open(pkl_path, "rb") as f:
        data = pickle.load(f, encoding="latin1")
    return data

In [9]:
# ─────────────────────────────────────────────
# HELPER: Downsample a signal to 1 Hz (1-sec intervals)
# by taking the mean of every window
# ─────────────────────────────────────────────
def downsample_mean(signal: np.ndarray, fs: int) -> np.ndarray:
    """
    Average every `fs` samples to produce a 1-Hz signal.
    Trims the tail so length is exactly divisible by fs.
    """
    n_seconds = len(signal) // fs
    trimmed    = signal[:n_seconds * fs]
    resampled  = trimmed.reshape(n_seconds, fs).mean(axis=1)
    return resampled

In [10]:
# ─────────────────────────────────────────────
# HELPER: Bandpass filter for BVP signal
# ─────────────────────────────────────────────
def bandpass_filter(signal: np.ndarray, fs: int,
                    lowcut: float = 0.5, highcut: float = 4.0) -> np.ndarray:
    """Apply a Butterworth bandpass filter (default: 0.5–4 Hz for BVP/HR)."""
    nyq = fs / 2.0
    low  = lowcut  / nyq
    high = highcut / nyq
    b, a = butter(3, [low, high], btype="band")
    return filtfilt(b, a, signal)

In [11]:
# ─────────────────────────────────────────────
# HELPER: Compute HR (BPM) from BVP in 1-sec windows
# ─────────────────────────────────────────────
def compute_hr_from_bvp(bvp: np.ndarray, fs: int = FS_BVP) -> np.ndarray:
    """
    Derive Heart Rate (BPM) from BVP signal.
    Steps:
      1. Bandpass filter the BVP (0.5–4 Hz covers 30–240 BPM)
      2. Detect peaks (heartbeats)
      3. Compute instantaneous HR per second window
    Returns a 1-Hz HR array in BPM.
    """
    # Filter BVP
    bvp_filtered = bandpass_filter(bvp, fs, lowcut=0.5, highcut=4.0)

    # Detect peaks (R-peaks / pulse peaks)
    # min_distance = 0.4 sec apart (max ~150 BPM)
    min_dist = int(fs * 0.4)
    peaks, _ = find_peaks(bvp_filtered, distance=min_dist)

    n_seconds = len(bvp) // fs
    hr_per_second = np.full(n_seconds, np.nan)

    for sec in range(n_seconds):
        start_sample = sec * fs
        end_sample   = start_sample + fs
        # Peaks that fall in this 1-second window
        peaks_in_window = peaks[(peaks >= start_sample) & (peaks < end_sample)]
        if len(peaks_in_window) >= 1:
            # Use a wider window (±2 sec) around this second to get IBI
            wider_start = max(0, start_sample - fs)
            wider_end   = min(len(bvp), end_sample + fs)
            peaks_wider = peaks[(peaks >= wider_start) & (peaks < wider_end)]
            if len(peaks_wider) >= 2:
                ibis = np.diff(peaks_wider) / fs          # Inter-beat intervals in seconds
                mean_ibi = np.mean(ibis)
                hr_per_second[sec] = 60.0 / mean_ibi      # Convert to BPM

    # Fill NaN gaps with forward/backward fill
    hr_series = pd.Series(hr_per_second).fillna(method="ffill").fillna(method="bfill")
    return hr_series.values

In [12]:
# ─────────────────────────────────────────────
# MAIN: Extract features for one subject
# ─────────────────────────────────────────────
def extract_subject_features(subject_id: int) -> pd.DataFrame:
    """
    Load raw data, extract and synchronize EDA, TEMP, HR.
    Returns a DataFrame with columns: [subject, second, EDA, TEMP, HR, label]
    filtered to Baseline (1) and Stress (2) only.
    """
    print(f"\n[Subject S{subject_id}] Loading data...")
    data = load_subject(subject_id)
    if data is None:
        return pd.DataFrame()

    # ── Chest signals (700 Hz) ──────────────────
    chest = data["signal"]["chest"]
    eda_chest  = chest["EDA"].flatten()     # Shape: (N,)
    temp_chest = chest["Temp"].flatten()    # Shape: (N,)
    labels_raw = data["label"].flatten()    # Shape: (N,) at 700 Hz

    # ── Wrist BVP (64 Hz) ───────────────────────
    bvp = data["signal"]["wrist"]["BVP"].flatten()

    print(f"  Chest EDA shape  : {eda_chest.shape}  @ {FS_CHEST} Hz")
    print(f"  Chest TEMP shape : {temp_chest.shape} @ {FS_CHEST} Hz")
    print(f"  Wrist BVP shape  : {bvp.shape}        @ {FS_BVP} Hz")
    print(f"  Labels shape     : {labels_raw.shape}  @ {FS_CHEST} Hz")

    # ── Downsample chest signals to 1 Hz ────────
    print("  Downsampling chest signals to 1 Hz...")
    eda_1hz   = downsample_mean(eda_chest,  FS_CHEST)
    temp_1hz  = downsample_mean(temp_chest, FS_CHEST)
    labels_1hz = downsample_mean(labels_raw, FS_CHEST)
    # Round labels back to integers after averaging
    labels_1hz = np.round(labels_1hz).astype(int)

    # ── Compute HR from BVP at 1 Hz ─────────────
    print("  Computing HR from BVP...")
    hr_1hz = compute_hr_from_bvp(bvp, fs=FS_BVP)

    # ── Align all signals to the shortest length ─
    min_len = min(len(eda_1hz), len(temp_1hz), len(labels_1hz), len(hr_1hz))
    eda_1hz    = eda_1hz[:min_len]
    temp_1hz   = temp_1hz[:min_len]
    labels_1hz = labels_1hz[:min_len]
    hr_1hz     = hr_1hz[:min_len]

    print(f"  Aligned length   : {min_len} seconds (~{min_len/60:.1f} minutes)")

    # ── Build DataFrame ──────────────────────────
    df = pd.DataFrame({
        "subject" : subject_id,
        "second"  : np.arange(min_len),
        "EDA"     : eda_1hz,
        "TEMP"    : temp_1hz,
        "HR"      : hr_1hz,
        "label"   : labels_1hz
    })

    # ── Filter: Keep only Baseline (1) and Stress (2) ──
    df = df[df["label"].isin([LABEL_BASELINE, LABEL_STRESS])].copy()

    # ── Remap labels: Baseline=0, Stress=1 (binary) ──
    df["label"] = df["label"].map({LABEL_BASELINE: 0, LABEL_STRESS: 1})

    baseline_count = (df["label"] == 0).sum()
    stress_count   = (df["label"] == 1).sum()
    print(f"  Baseline seconds : {baseline_count}")
    print(f"  Stress seconds   : {stress_count}")

    return df

In [13]:
# ─────────────────────────────────────────────
# RUN
# ─────────────────────────────────────────────
if __name__ == "__main__":
    print("=" * 60)
    print("=" * 60)
    print(f"  Data path : {WESAD_PATH}")
    print(f"  Output    : {OUTPUT_CSV}")
    print(f"  Subjects  : {SUBJECT_IDS}")
    print("=" * 60)

    all_dfs = []

    for sid in SUBJECT_IDS:
        df_subj = extract_subject_features(sid)
        if not df_subj.empty:
            all_dfs.append(df_subj)

    if not all_dfs:
        print("\n[ERROR] No data was loaded. Check your WESAD_PATH.")
    else:
        # Combine all subjects
        combined = pd.concat(all_dfs, ignore_index=True)

        # ── Basic sanity checks ──────────────────
        print("\n" + "=" * 60)
        print("  FINAL DATASET SUMMARY")
        print("=" * 60)
        print(f"  Total samples    : {len(combined)}")
        print(f"  Subjects loaded  : {combined['subject'].nunique()}")
        print(f"  Baseline samples : {(combined['label'] == 0).sum()}")
        print(f"  Stress samples   : {(combined['label'] == 1).sum()}")
        print(f"\n  Feature stats:")
        print(combined[["EDA", "TEMP", "HR"]].describe().round(3).to_string())

        # ── Check for NaN values ─────────────────
        nan_counts = combined[["EDA", "TEMP", "HR"]].isna().sum()
        print(f"\n  NaN counts:")
        print(nan_counts.to_string())
        if nan_counts.sum() > 0:
            print("  [INFO] Dropping rows with NaN values...")
            combined.dropna(subset=["EDA", "TEMP", "HR"], inplace=True)
            print(f"  Remaining samples: {len(combined)}")

        # ── Save to CSV ──────────────────────────
        combined.to_csv(OUTPUT_CSV, index=False)
        print(f"\n  [SUCCESS] Saved to: {OUTPUT_CSV}")
        print("=" * 60)
        print("\n  Columns in output:")
        print("    subject  -> Subject ID (e.g. 2, 3, ...)")
        print("    second   -> Time index within subject session")
        print("    EDA      -> Electrodermal Activity (chest, µS)")
        print("    TEMP     -> Skin Temperature (chest, °C)")
        print("    HR       -> Heart Rate (wrist BVP, BPM)")
        print("    label    -> 0=Baseline, 1=Stress  (binary)")
        print("=" * 60)



  Data path : /content/WESAD/WESAD
  Output    : wesad_features.csv
  Subjects  : [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]

[Subject S2] Loading data...
  Chest EDA shape  : (4255300,)  @ 700 Hz
  Chest TEMP shape : (4255300,) @ 700 Hz
  Wrist BVP shape  : (389056,)        @ 64 Hz
  Labels shape     : (4255300,)  @ 700 Hz
  Downsampling chest signals to 1 Hz...
  Computing HR from BVP...


/tmp/ipython-input-25773/1839821234.py:40: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  hr_series = pd.Series(hr_per_second).fillna(method="ffill").fillna(method="bfill")


  Aligned length   : 6079 seconds (~101.3 minutes)
  Baseline seconds : 1147
  Stress seconds   : 619

[Subject S3] Loading data...
  Chest EDA shape  : (4545100,)  @ 700 Hz
  Chest TEMP shape : (4545100,) @ 700 Hz
  Wrist BVP shape  : (415552,)        @ 64 Hz
  Labels shape     : (4545100,)  @ 700 Hz
  Downsampling chest signals to 1 Hz...
  Computing HR from BVP...
  Aligned length   : 6493 seconds (~108.2 minutes)
  Baseline seconds : 1145
  Stress seconds   : 640

[Subject S4] Loading data...
  Chest EDA shape  : (4496100,)  @ 700 Hz
  Chest TEMP shape : (4496100,) @ 700 Hz
  Wrist BVP shape  : (411072,)        @ 64 Hz
  Labels shape     : (4496100,)  @ 700 Hz
  Downsampling chest signals to 1 Hz...
  Computing HR from BVP...
  Aligned length   : 6423 seconds (~107.0 minutes)
  Baseline seconds : 1164
  Stress seconds   : 637

[Subject S5] Loading data...
  Chest EDA shape  : (4380600,)  @ 700 Hz
  Chest TEMP shape : (4380600,) @ 700 Hz
  Wrist BVP shape  : (400512,)        @ 64 Hz